In [1]:
# generate training and testing data
# partition the data into L nodes by features
# ridge regression on L nodes
# average the results from each node
# get MSE

In [2]:
import numpy as np 
from GenerateTrainingData import GenerateTrainingData
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [3]:
STD_DEV = 0.31622776601 # sqrt(0.1)
TRAIN_SIZE = 1000
TEST_SIZE = 10000
NUM_NODES = 10

data_generator = GenerateTrainingData(num_features=101, STD_DEV=STD_DEV)
A_mat_train, B_vec_train, C_mat_train = data_generator.generate_data(num_data_points=TRAIN_SIZE)
A_mat_test, B_vec_test, C_mat_test = data_generator.generate_data(num_data_points=TEST_SIZE)

In [4]:
def train_ridge_models(C_mat_partitioned: list[np.ndarray], B_vec: np.ndarray, alpha: float = 1.0):
    models = []
    
    for i, partition in enumerate(C_mat_partitioned):
        model = Ridge(alpha=alpha)
        model.fit(partition, B_vec)
        
        models.append(model)
        print(f"Model {i+1}/{len(C_mat_partitioned)} trained")
        
    return models

In [5]:
def evaluate_partitioned_models(models, C_mat_test, B_vec_test, num_nodes):
    C_mat_test_partitioned = np.array_split(C_mat_test, num_nodes, axis=1)
    
    # each node predicts in a vacuum then average the result
    all_predictions = []
    for i, model in enumerate(models):
        partition_preds = model.predict(C_mat_test_partitioned[i])
        all_predictions.append(partition_preds)
    
    avg_predictions = np.mean(all_predictions, axis=0)
    
    mse = mean_squared_error(B_vec_test, avg_predictions)
    
    return avg_predictions, mse

In [6]:
C_mat_train_partitioned = np.array_split(C_mat_train, NUM_NODES, axis=1)
trained_models = train_ridge_models(C_mat_train_partitioned, B_vec_train, alpha=1.0)

Model 1/10 trained
Model 2/10 trained
Model 3/10 trained
Model 4/10 trained
Model 5/10 trained
Model 6/10 trained
Model 7/10 trained
Model 8/10 trained
Model 9/10 trained
Model 10/10 trained


In [7]:
avg_preds, final_mse = evaluate_partitioned_models(
    trained_models, 
    C_mat_test, 
    B_vec_test, 
    NUM_NODES
)

print(f"Final Mean Squared Error: {final_mse:.4f}")

Final Mean Squared Error: 1.9037


In [8]:
NUM_NODES = 5

C_mat_train_partitioned = np.array_split(C_mat_train, NUM_NODES, axis=1)
trained_models = train_ridge_models(C_mat_train_partitioned, B_vec_train, alpha=0.5)
avg_preds, final_mse = evaluate_partitioned_models(
    trained_models, 
    C_mat_test, 
    B_vec_test, 
    NUM_NODES
)

print(f"Final Mean Squared Error: {final_mse:.4f}")

Model 1/5 trained
Model 2/5 trained
Model 3/5 trained
Model 4/5 trained
Model 5/5 trained
Final Mean Squared Error: 1.2889


In [9]:
NUM_NODES = 1
C_mat_train_partitioned = np.array_split(C_mat_train, NUM_NODES, axis=1)
trained_models = train_ridge_models(C_mat_train_partitioned, B_vec_train, alpha=0.5)
avg_preds, final_mse = evaluate_partitioned_models(
    trained_models, 
    C_mat_test, 
    B_vec_test, 
    NUM_NODES
)

print(f"Final Mean Squared Error: {final_mse:.4f}")

Model 1/1 trained
Final Mean Squared Error: 0.5546
